In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Load model from HuggingFace Hub
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-mpnet-base-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-mpnet-base-v2')

/Users/nyuad/Desktop/capstone/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
agent_sentences = ["A man is eating pizza", "A cat is sleeping", "It is raining outside"]
gt_claims = ["A man eats something", "A dog is running", "The weather is dry and sunny"]

def encode(sentences):
    encoded = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        output = model(**encoded)
    embeddings = mean_pooling(output, encoded['attention_mask'])
    return F.normalize(embeddings, p=2, dim=1)

agent_embeds = encode(agent_sentences)
gt_embeds = encode(gt_claims)

In [3]:
# Full cosine similarity matrix: [n_agent, n_gt]
sim_matrix = agent_embeds @ gt_embeds.T

print("Cosine Similarity Matrix (agent sentences x GT claims):\n")
for i, agent_s in enumerate(agent_sentences):
    for j, gt_c in enumerate(gt_claims):
        print(f"  [{i},{j}] {sim_matrix[i,j].item():.4f}  |  \"{agent_s}\"  <->  \"{gt_c}\"")
    print()

print("Best GT match per agent sentence:")
for i, agent_s in enumerate(agent_sentences):
    best_j = sim_matrix[i].argmax().item()
    print(f"  \"{agent_s}\"  ->  \"{gt_claims[best_j]}\"  (sim={sim_matrix[i, best_j].item():.4f})")

print(f"\nAverage best-match similarity: {sim_matrix.max(dim=1).values.mean().item():.4f}")

Cosine Similarity Matrix (agent sentences x GT claims):

  [0,0] 0.5129  |  "A man is eating pizza"  <->  "A man eats something"
  [0,1] -0.0506  |  "A man is eating pizza"  <->  "A dog is running"
  [0,2] -0.0163  |  "A man is eating pizza"  <->  "The weather is dry and sunny"

  [1,0] -0.0182  |  "A cat is sleeping"  <->  "A man eats something"
  [1,1] -0.0678  |  "A cat is sleeping"  <->  "A dog is running"
  [1,2] 0.0590  |  "A cat is sleeping"  <->  "The weather is dry and sunny"

  [2,0] 0.0087  |  "It is raining outside"  <->  "A man eats something"
  [2,1] 0.0975  |  "It is raining outside"  <->  "A dog is running"
  [2,2] 0.2995  |  "It is raining outside"  <->  "The weather is dry and sunny"

Best GT match per agent sentence:
  "A man is eating pizza"  ->  "A man eats something"  (sim=0.5129)
  "A cat is sleeping"  ->  "The weather is dry and sunny"  (sim=0.0590)
  "It is raining outside"  ->  "The weather is dry and sunny"  (sim=0.2995)

Average best-match similarity: 0.2905

In [4]:
import json
from pathlib import Path
from nltk.tokenize import sent_tokenize
import numpy as np

EXPERIMENT_PATH = '/Users/nyuad/Desktop/capstone/experiments/baseline_social/run_0029/experiment.json'
experiment_data = json.loads(Path(EXPERIMENT_PATH).read_text())

GT = ("University media crews record interviews and highlights across the career fair floor. "
      "Recruiters distribute tote bags, notebooks, tech swag, and follow-up instructions. "
      "Startups schedule lunchtime pitch sessions while international students receive visa guidance. "
      "Whiteboards and dashboards track interview slots, attendance numbers, and recruiter availability. "
      "Nonprofits share mission-driven roles beside corporate employers, emphasizing varied career paths. "
      "Students upload resumes at digital kiosks tied to the fair database. "
      "The dean thanks recruiters while career center staff log metrics on laptops. "
      "Chimes announce the final hour as recruiters encourage personalized email follow-ups. "
      "Volunteers pack booths and collect leftover brochures at the end of the day. "
      "Students depart with stacks of business cards, survey links, and notes on responsive recruiters. "
      "Friends debrief on the quad, comparing impressions of company cultures and enthusiasm. "
      "Post-event planning focuses on scheduling interviews and sustaining new connections.")

gt_claims = sent_tokenize(GT)
print(f"{len(gt_claims)} GT claims loaded")

12 GT claims loaded


In [ ]:
AGENT_ID = "0"

agent_data = experiment_data['agents'][AGENT_ID]
summaries = list(agent_data['summaries'].values())
latest_summary = summaries[-1]

agent_sentences = sent_tokenize(latest_summary)
print(f"Agent {AGENT_ID} — {len(agent_sentences)} sentences in latest summary\n")
print(latest_summary)

Agent 0 — 7 sentences in latest summary

At the university gym, a busy career fair attracts international students eager to connect with employers and seek advice on visa sponsorship requirements, clutching resumes inside branded folders. Event staff scan registrations and distribute lanyards, while career center personnel log attendance metrics on laptops. Dozens of booths adorned with colorful banners represent various sectors, including consulting, finance, nonprofits, and a robotics startup showcasing a flying drone. Finance firms offer QR codes for immediate internship applications, alongside a cybersecurity company hosting a password cracking challenge and a data science firm displaying a live dashboard of attendance metrics. Career counselors redirect unsure students toward applicable industries while also guiding students in adjoining classrooms during workshops on networking etiquette. Monitors display a post-event survey link, and enthusiastic recruiters engage students, offe

In [ ]:
agent_embeds = encode(agent_sentences)
gt_embeds = encode(gt_claims)

# [n_agent_sentences, n_gt_claims]
sim_matrix = agent_embeds @ gt_embeds.T
sim_np = sim_matrix.numpy()

print(f"Similarity matrix shape: {sim_np.shape}  (agent sentences x GT claims)\n")

print("Best GT match per agent sentence:")
for i, sent in enumerate(agent_sentences):
    best_j = sim_np[i].argmax()
    print(f"  S{i}: {sim_np[i, best_j]:.4f}  →  \"{gt_claims[best_j][:80]}\"")
    print(f"       \"{sent[:80]}\"")

best_per_gt = sim_np.max(axis=0)
print(f"\nBest agent match per GT claim:")
for j, claim in enumerate(gt_claims):
    best_i = sim_np[:, j].argmax()
    print(f"  F{j}: {best_per_gt[j]:.4f}  ←  S{best_i}")
    print(f"       \"{claim[:80]}\"")

avg_best = best_per_gt.mean()
covered = (best_per_gt > 0.5).sum()
print(f"\nAverage best-per-GT-claim similarity: {avg_best:.4f}")
print(f"GT claims covered (sim > 0.5): {covered}/{len(gt_claims)}")

Similarity matrix shape: (7, 12)  (agent sentences x GT claims)

Best GT match per agent sentence:
  S0: 0.5722  →  "Students depart with stacks of business cards, survey links, and notes on respon"
       "At the university gym, a busy career fair attracts international students eager "
  S1: 0.6069  →  "The dean thanks recruiters while career center staff log metrics on laptops."
       "Event staff scan registrations and distribute lanyards, while career center pers"
  S2: 0.4650  →  "Volunteers pack booths and collect leftover brochures at the end of the day."
       "Dozens of booths adorned with colorful banners represent various sectors, includ"
  S3: 0.5188  →  "Students upload resumes at digital kiosks tied to the fair database."
       "Finance firms offer QR codes for immediate internship applications, alongside a "
  S4: 0.4771  →  "Students depart with stacks of business cards, survey links, and notes on respon"
       "Career counselors redirect unsure students toward app

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

def plot_sim_heatmap(matrix, row_labels, col_labels, title="Cosine Similarity", max_label=72):
    trunc = lambda s: s[:max_label].rsplit(" ", 1)[0] + "…" if len(s) > max_label else s
    rl = [f"S{i}: {trunc(s)}" for i, s in enumerate(row_labels)]
    cl = [f"F{j}: {trunc(s)}" for j, s in enumerate(col_labels)]

    n_r, n_c = matrix.shape
    cell_w = max(1.4, min(2.2, 20.0 / n_c))
    cell_h = max(0.6, min(1.2, 14.0 / n_r))
    fig, ax = plt.subplots(figsize=(max(10, n_c * cell_w + 4), max(4, n_r * cell_h + 3)),
                           facecolor="#0d1117")
    ax.set_facecolor("#0d1117")

    cmap = LinearSegmentedColormap.from_list("br", ["#1e3a5f", "#3b7dd8", "#f0f0f0", "#e8644a", "#c0392b"])
    im = ax.imshow(matrix, cmap=cmap, aspect="auto", vmin=0, vmax=1)

    for i in range(n_r):
        for j in range(n_c):
            v = matrix[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=max(6, min(9, 100 // max(n_r, n_c))),
                    color="white" if v < 0.35 or v > 0.75 else "#1a1a2e", fontweight="bold")

    ax.set_xticks(range(n_c)); ax.set_xticklabels(cl, rotation=40, ha="right", fontsize=7, color="#c9d1d9")
    ax.set_yticks(range(n_r)); ax.set_yticklabels(rl, fontsize=7, color="#c9d1d9")
    ax.set_title(title, fontsize=14, color="#fff", fontweight="bold", pad=12)
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.04).ax.tick_params(colors="#8b949e", labelsize=8)
    for s in ax.spines.values(): s.set_visible(False)
    plt.tight_layout()
    plt.show()

plot_sim_heatmap(sim_np, agent_sentences, gt_claims, title=f"Cosine Similarity — Agent {AGENT_ID}")